# SRResNet training: Sentinel-2 (10 m) -> PlanetScope (3 m), 4x

**Before running:**
1. Right side panel -> **Add Input** -> attach the dataset that contains `satellite_sr_dataset/HR` and `LR` (only read, never modified).
2. **Session options -> Accelerator** -> choose a **GPU** (T4 / P100).
3. **Run All**. Internet is not required.

No pretrained checkpoint exists, so this trains **from scratch**. Checkpoints go to `/kaggle/working/sr_output/srresnet_checkpoints/`.

**What this version fixes**
- `TypeError: __bases__ assignment` -> `PairedTiffDataset` now subclasses `torch.utils.data.Dataset` normally.
- LR and HR were not a 4x pair (HR 46x76 @ 3 m UTM, LR 27x40 @ ~10 m lat/lon, larger area). LR is now reprojected onto the HR footprint at 12 m, so LR 11x19 -> HR 44x76 is an exact 4x pair.
- Band mismatch: the old code read bands 1-4 of both files (LR = Blue, Green, Red, NIR but HR = Coastal blue, Blue, Green I, Green). Now both are Blue, Green, Red, NIR.
- HR nodata (0, about a third of every Planet tile) and LR `-inf` are masked out of the loss.
- Prints a bicubic baseline so you can see whether the model actually beats simple upsampling.


In [ ]:
# Cell 1: environment check (installs rasterio only if it is missing)
import importlib, subprocess, sys

for module, pip_name in [("numpy", "numpy"), ("torch", "torch"), ("rasterio", "rasterio"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(module)
    except ImportError:
        print(f"{module} missing -> installing {pip_name} (needs Internet ON)")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

import torch, rasterio
print("torch", torch.__version__, "| rasterio", rasterio.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "none - enable one under Session options -> Accelerator")

In [ ]:
# Cell 2: data preparation, model and training code
"""Train or fine-tune a four-band (Blue, Green, Red, NIR) SRResNet on co-registered
Sentinel-2 (LR) / PlanetScope (HR) GeoTIFF pairs.

Expected input layout (left untouched, only read):

    satellite_sr_dataset/
        HR/**/*.tif   PlanetScope SuperDove, 8 bands, 3 m, UTM (EPSG:32643), uint16
        LR/**/*.tif   GEE Sentinel-2 export, 8 bands (B2,B3,B4,B8,B11,NDVI,NDRE,SI),
                      ~10 m, EPSG:4326, float reflectance, -inf nodata

The raw files are NOT a ready 4x pair: the LR tile covers a larger area, sits in a
different CRS and has a different band order. ``prepare_pair`` therefore:

1. picks the same physical bands from both sensors (Blue, Green, Red, NIR),
2. crops HR so both sides are divisible by 4 (46x76 -> 44x76),
3. reprojects the Sentinel-2 bands onto the HR footprint at 4x the HR pixel size
   (3 m * 4 = 12 m -> 11x19 LR grid), so LR and HR cover exactly the same ground,
4. builds a validity mask (HR nodata = 0, LR nodata = non-finite) used by the loss.
"""

from __future__ import annotations

import argparse
import os
import random
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as functional
from torch.utils.data import DataLoader, Dataset

FACTOR = 4
CHANNELS = 4
SCALE_FACTOR = 10000.0
BAND_NAMES = ("blue", "green", "red", "nir")
# GEE export order is B2, B3, B4, B8, B11, NDVI, NDRE, SI -> Blue, Green, Red, NIR = 1..4
LR_BANDS = (1, 2, 3, 4)
# PlanetScope SuperDove fallback if band descriptions are missing:
# coastal_blue, blue, green_i, green, yellow, red, rededge, nir
HR_BANDS_FALLBACK = (2, 4, 6, 8)
MIN_VALID_FRACTION = 0.5


def normalize(array: np.ndarray) -> np.ndarray:
    """Convert reflectance or Sentinel-2 / Planet digital numbers to 0..1 (non-finite kept as NaN)."""
    raw = np.asarray(array)
    values = raw.astype(np.float32)
    if raw.dtype == np.uint8:
        values = values / 255.0
    elif np.issubdtype(raw.dtype, np.integer):
        values = values / SCALE_FACTOR  # reflectance * 10000
    else:
        finite = np.isfinite(values)
        if finite.any() and float(np.nanmax(np.where(finite, values, np.nan))) > 20:
            values = values / SCALE_FACTOR
    values[~np.isfinite(values)] = np.nan
    return np.clip(values, 0, 1, where=np.isfinite(values), out=values)


def hr_band_indexes(src: Any) -> tuple[int, ...]:
    names = [(d or "").lower() for d in src.descriptions]
    if all(name in names for name in BAND_NAMES):
        return tuple(names.index(name) + 1 for name in BAND_NAMES)
    return HR_BANDS_FALLBACK


def discover_dataset(input_root: str | Path) -> tuple[Path, Path]:
    """Find the existing satellite_sr_dataset/HR and LR directories."""
    root = Path(input_root)
    if not root.exists():
        raise FileNotFoundError(f"{root} does not exist. Attach the dataset via Add Input.")
    candidates: list[tuple[Path, Path, Path]] = []
    for dirpath, dirnames, _ in os.walk(root):
        lowered = {name.lower(): name for name in dirnames}
        if "hr" in lowered and "lr" in lowered:
            base = Path(dirpath)
            candidates.append((base, base / lowered["hr"], base / lowered["lr"]))
            dirnames[:] = []
    if not candidates:
        raise FileNotFoundError(f"Could not locate satellite_sr_dataset/HR and LR below {root}")
    candidates.sort(key=lambda item: item[0].name != "satellite_sr_dataset")
    base, hr_dir, lr_dir = candidates[0]
    print(f"Using dataset folder: {base}")
    return hr_dir, lr_dir


def paired_paths(hr_dir: Path, lr_dir: Path) -> list[tuple[Path, Path]]:
    """Return (LR, HR) TIFFs that share a basename (same farm + date)."""
    def index(folder: Path) -> dict[str, Path]:
        result: dict[str, Path] = {}
        for path in sorted(folder.rglob("*")):
            if path.is_file() and path.suffix.lower() in {".tif", ".tiff"}:
                if path.stem in result:
                    raise ValueError(f"Duplicate TIFF basename in {folder}: {path.stem}")
                result[path.stem] = path
        return result

    hr, lr = index(hr_dir), index(lr_dir)
    names = sorted(set(hr) & set(lr))
    if not names:
        raise ValueError("No matching HR/LR TIFF basenames were found")
    print(f"HR TIFFs: {len(hr)} | LR TIFFs: {len(lr)} | matched by date: {len(names)}")
    return [(lr[name], hr[name]) for name in names]


def prepare_pair(lr_path: Path, hr_path: Path) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Co-register one Sentinel-2 / Planet pair.

    Returns LR (4, h, w), HR (4, 4h, 4w) in 0..1 and an HR-sized validity mask (1, 4h, 4w).
    """
    import rasterio
    from rasterio.transform import Affine
    from rasterio.warp import Resampling, reproject

    with rasterio.open(hr_path) as hr_src:
        if hr_src.crs is None:
            raise ValueError(f"{hr_path.name}: HR has no CRS")
        bands = hr_band_indexes(hr_src)
        hr_raw = hr_src.read(bands)
        hr_crs, hr_t = hr_src.crs, hr_src.transform
        hr_nodata = hr_src.nodata if hr_src.nodata is not None else 0
    height = (hr_raw.shape[1] // FACTOR) * FACTOR
    width = (hr_raw.shape[2] // FACTOR) * FACTOR
    hr_raw = hr_raw[:, :height, :width]
    hr_valid = np.all(hr_raw != hr_nodata, axis=0)
    hr = normalize(hr_raw)

    lr_t = Affine(hr_t.a * FACTOR, hr_t.b, hr_t.c, hr_t.d, hr_t.e * FACTOR, hr_t.f)
    lr = np.full((CHANNELS, height // FACTOR, width // FACTOR), np.nan, dtype=np.float32)
    with rasterio.open(lr_path) as lr_src:
        if lr_src.crs is None or lr_src.count < max(LR_BANDS):
            raise ValueError(f"{lr_path.name}: LR needs a CRS and {max(LR_BANDS)} bands")
        src = lr_src.read(LR_BANDS).astype(np.float32)
        src[~np.isfinite(src)] = np.nan
        if not np.isfinite(src).any():
            raise ValueError(f"{lr_path.name}: LR contains no finite pixels")
        reproject(
            source=src, destination=lr,
            src_transform=lr_src.transform, src_crs=lr_src.crs, src_nodata=np.nan,
            dst_transform=lr_t, dst_crs=hr_crs, dst_nodata=np.nan,
            resampling=Resampling.bilinear,
        )
    lr = normalize(lr)
    lr_valid = np.all(np.isfinite(lr), axis=0)
    valid = hr_valid & np.all(np.isfinite(hr), axis=0)
    valid &= np.repeat(np.repeat(lr_valid, FACTOR, axis=0), FACTOR, axis=1)
    if valid.mean() < MIN_VALID_FRACTION:
        raise ValueError(f"{hr_path.name}: only {valid.mean():.0%} valid pixels")
    lr = np.nan_to_num(lr, nan=0.0)
    hr = np.nan_to_num(hr, nan=0.0)
    return lr, hr, valid[None].astype(np.float32)


def load_pairs(pairs: list[tuple[Path, Path]]) -> list[tuple[str, np.ndarray, np.ndarray, np.ndarray]]:
    """Prepare every pair once (the dataset is tiny) and skip unusable ones."""
    samples, skipped = [], []
    for lr_path, hr_path in pairs:
        try:
            samples.append((hr_path.stem, *prepare_pair(lr_path, hr_path)))
        except Exception as error:
            skipped.append(f"{hr_path.stem}: {error}")
    print(f"usable pairs: {len(samples)} | skipped: {len(skipped)}")
    for line in skipped:
        print("  skipped", line)
    if samples:
        _, lr, hr, mask = samples[0]
        print(f"LR shape {lr.shape} | HR shape {hr.shape} | valid {mask.mean():.0%}")
    return samples


class PairedTiffDataset(Dataset):
    """In-memory dataset of co-registered (LR, HR, mask) tensors."""

    def __init__(self, samples, augment: bool = False):
        super().__init__()
        self.samples = samples
        self.augment = augment

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int):
        _, lr, hr, mask = self.samples[index]
        if self.augment:
            if random.random() < 0.5:
                lr, hr, mask = lr[:, :, ::-1], hr[:, :, ::-1], mask[:, :, ::-1]
            if random.random() < 0.5:
                lr, hr, mask = lr[:, ::-1, :], hr[:, ::-1, :], mask[:, ::-1, :]
        return (torch.from_numpy(lr.copy()), torch.from_numpy(hr.copy()),
                torch.from_numpy(mask.copy()))


class ResidualBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1), nn.BatchNorm2d(channels),
            nn.PReLU(channels),
            nn.Conv2d(channels, channels, 3, padding=1), nn.BatchNorm2d(channels),
        )

    def forward(self, x):
        return x + self.body(x)


class SRResNet(nn.Module):
    """4-channel SRResNet with two 2x pixel-shuffle blocks (4x total)."""

    def __init__(self, features: int = 64, blocks: int = 16):
        super().__init__()
        self.head = nn.Sequential(nn.Conv2d(CHANNELS, features, 9, padding=4), nn.PReLU(features))
        self.residual = nn.Sequential(
            *[ResidualBlock(features) for _ in range(blocks)],
            nn.Conv2d(features, features, 3, padding=1), nn.BatchNorm2d(features),
        )
        self.upsample = nn.Sequential(
            nn.Conv2d(features, features * 4, 3, padding=1), nn.PixelShuffle(2), nn.PReLU(features),
            nn.Conv2d(features, features * 4, 3, padding=1), nn.PixelShuffle(2), nn.PReLU(features),
        )
        self.tail = nn.Sequential(
            nn.Conv2d(features, features, 3, padding=1), nn.PReLU(features),
            nn.Conv2d(features, CHANNELS, 9, padding=4), nn.Sigmoid(),
        )

    def forward(self, x):
        shallow = self.head(x)
        return self.tail(self.upsample(self.residual(shallow) + shallow))


def build_model() -> SRResNet:
    return SRResNet()


def masked_l1(prediction, target, mask):
    return (torch.abs(prediction - target) * mask).sum() / (mask.sum() * prediction.shape[1]).clamp_min(1)


def split_samples(samples, validation_fraction: float, seed: int):
    if len(samples) < 2:
        raise ValueError("At least two usable HR/LR pairs are required")
    if not 0 < validation_fraction < 1:
        raise ValueError("validation_fraction must be between 0 and 1")
    shuffled = list(samples)
    random.Random(seed).shuffle(shuffled)
    size = min(max(1, round(len(shuffled) * validation_fraction)), len(shuffled) - 1)
    return shuffled[size:], shuffled[:size]


def load_checkpoint(model: nn.Module, checkpoint: Path, device) -> None:
    if not checkpoint.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint}. Use from_scratch=True if none exists.")
    payload = torch.load(checkpoint, map_location=device)
    state = payload.get("model_state_dict", payload.get("state_dict", payload))
    if not isinstance(state, dict):
        raise ValueError(f"Unsupported checkpoint format: {checkpoint}")
    model.load_state_dict(state, strict=True)


def bicubic_baseline(loader, device) -> float:
    """Masked L1 of plain bicubic upsampling: the model has to beat this."""
    total, count = 0.0, 0
    for lr, hr, mask in loader:
        lr, hr, mask = lr.to(device), hr.to(device), mask.to(device)
        up = functional.interpolate(lr, scale_factor=FACTOR, mode="bicubic", align_corners=False).clamp(0, 1)
        total += float(masked_l1(up, hr, mask)) * len(lr)
        count += len(lr)
    return total / max(count, 1)


def run_epoch(model, loader, optimizer, device) -> float:
    training = optimizer is not None
    model.train(training)
    total, count = 0.0, 0
    for lr, hr, mask in loader:
        lr, hr, mask = lr.to(device), hr.to(device), mask.to(device)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            loss = masked_l1(model(lr), hr, mask)
        if training:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        total += float(loss.detach()) * len(lr)
        count += len(lr)
    return total / max(count, 1)


def train(args: argparse.Namespace) -> dict[str, float]:
    torch.manual_seed(args.seed)
    random.seed(args.seed)
    np.random.seed(args.seed)
    hr_dir, lr_dir = discover_dataset(args.input_root)
    samples = load_pairs(paired_paths(hr_dir, lr_dir))
    train_samples, validation_samples = split_samples(samples, args.validation_fraction, args.seed)
    print(f"train pairs: {len(train_samples)} | validation pairs: {len(validation_samples)}")
    device = torch.device(args.device if args.device != "auto"
                          else ("cuda" if torch.cuda.is_available() else "cpu"))
    print(f"device: {device}")
    model = build_model().to(device)
    if not args.from_scratch:
        if args.checkpoint is None:
            raise ValueError("Fine-tuning requires checkpoint=PATH, or set from_scratch=True.")
        load_checkpoint(model, Path(args.checkpoint), device)

    train_loader = DataLoader(PairedTiffDataset(train_samples, augment=True),
                              batch_size=args.batch_size, shuffle=True, num_workers=args.workers)
    validation_loader = DataLoader(PairedTiffDataset(validation_samples),
                                   batch_size=args.batch_size, shuffle=False, num_workers=args.workers)
    baseline = bicubic_baseline(validation_loader, device)
    print(f"bicubic baseline validation_l1={baseline:.6f} (lower is better; beat this)")
    optimizer = torch.optim.Adam(model.parameters(), lr=args.learning_rate)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)
    checkpoint_dir = Path(args.output_root) / "srresnet_checkpoints"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    best_loss = float("inf")
    for epoch in range(1, args.epochs + 1):
        train_loss = run_epoch(model, train_loader, optimizer, device)
        validation_loss = run_epoch(model, validation_loader, None, device)
        scheduler.step()
        improved = validation_loss < best_loss
        payload = {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch, "validation_l1": validation_loss,
            "channels": CHANNELS, "factor": FACTOR, "bands": BAND_NAMES,
        }
        torch.save(payload, checkpoint_dir / "srresnet_last.pt")
        if improved:
            best_loss = validation_loss
            torch.save(payload, checkpoint_dir / "srresnet_best.pt")
        if improved or epoch % args.log_every == 0 or epoch == 1:
            print(f"epoch {epoch:03d}/{args.epochs:03d} train_l1={train_loss:.6f} "
                  f"validation_l1={validation_loss:.6f}" + ("  * best" if improved else ""))
    print(f"best validation_l1={best_loss:.6f} vs bicubic {baseline:.6f}")
    return {"best_validation_l1": best_loss, "bicubic_validation_l1": baseline}


In [ ]:
# Cell 3: dataset sanity check (read-only; run before training)
hr_dir_check, lr_dir_check = discover_dataset("/kaggle/input")
samples_check = load_pairs(paired_paths(hr_dir_check, lr_dir_check))
shapes = {(lr.shape, hr.shape) for _, lr, hr, _ in samples_check}
print("distinct (LR, HR) shapes:", shapes)
assert len(shapes) == 1, "patches differ in size; batching would fail"
lr_all = np.stack([s[1] for s in samples_check]); hr_all = np.stack([s[2] for s in samples_check])
print("LR range", float(lr_all.min()), float(lr_all.max()), "| HR range", float(hr_all.min()), float(hr_all.max()))

In [ ]:
# Cell 4: train from scratch (no pretrained checkpoint available)
args = argparse.Namespace(
    input_root="/kaggle/input",
    output_root="/kaggle/working/sr_output",
    checkpoint=None,
    from_scratch=True,
    epochs=300,          # tiny patches: 300 epochs take a few minutes on a T4
    batch_size=8,
    learning_rate=2e-4,
    validation_fraction=0.2,
    seed=42,
    workers=0,
    device="auto",
    log_every=25,
)
metrics = train(args)
print(metrics)
print(sorted(str(p) for p in Path(args.output_root).rglob("*.pt")))

In [ ]:
# Cell 5: visual check on validation patches (false-colour RGB)
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model().to(device)
load_checkpoint(model, Path(args.output_root) / "srresnet_checkpoints" / "srresnet_best.pt", device)
model.eval()
_, validation_samples = split_samples(samples_check, args.validation_fraction, args.seed)

def rgb(x):  # x: (4, H, W) in Blue, Green, Red, NIR order
    img = np.stack([x[2], x[1], x[0]], -1)
    return np.clip(img / max(np.percentile(img, 98), 1e-6), 0, 1)

rows = validation_samples[:4]
fig, axes = plt.subplots(len(rows), 4, figsize=(14, 3 * len(rows)), squeeze=False)
for row, (name, lr, hr, mask) in zip(axes, rows):
    lr_t = torch.from_numpy(lr)[None].to(device)
    with torch.no_grad():
        sr = model(lr_t)[0].cpu().numpy()
        bic = functional.interpolate(lr_t, scale_factor=FACTOR, mode="bicubic").clamp(0, 1)[0].cpu().numpy()
    for ax, img, title in zip(row, [lr, bic, sr, hr * mask], ["LR 12 m", "bicubic", "SRResNet", "HR 3 m (masked)"]):
        ax.imshow(rgb(img), interpolation="nearest"); ax.set_title(f"{name}\n{title}", fontsize=8); ax.axis("off")
plt.tight_layout()
plt.savefig(Path(args.output_root) / "validation_preview.png", dpi=120)
plt.show()

To fine-tune later, set `from_scratch=False` and `checkpoint="/kaggle/working/sr_output/srresnet_checkpoints/srresnet_best.pt"` in Cell 4.
Download results from the **Output** tab (use **Save Version -> Save & Run All** to keep them).

**Note on inference:** the model expects LR input in Blue, Green, Red, NIR order on a 12 m grid. To run it on a raw GEE export, reproject bands 1-4 to 12 m first (see `prepare_pair`).